# CR Decomposition

**Idea**:
Split a matrix `A` into `C` (a subset of A's own columns — the
independent ones) and `R` (a small matrix of weights), such that
`A = C · R`. Every column left out of `C` can be exactly rebuilt as a
combination of `C`'s columns using the matching column of `R`.

**Why**:
Real datasets often have redundant features — e.g. a `Total_Cost`
column that's really just `Item_Price + Shipping_Fee` added together. CR
decomposition finds these automatically: the columns kept in `C` are provably
independent, and `R` shows exactly how every dropped column was built from
them.

**How**:
In this notebook:
1. Build a small dataset where one feature (`Total_Cost`) is deliberately
   redundant — it's just `Item_Price + Shipping_Fee`.
2. Compute the RREF of `A` with `sympy` — this reveals the pivot (independent)
   columns and the recipe matrix `R`.
3. Build `C` from A's actual pivot columns.
4. Flag every non-pivot column as redundant.
5. Verify `C @ R[:rank, :]` reconstructs `A` exactly.
6. Drop the redundant columns and keep the reduced dataset.

In [ ]:
!pip install sympy
import numpy as np
from sympy import Matrix

# 1. Create a dummy dataset (4 features, 5 samples)
# Feature 0: Random data
# Feature 1: Random data
# Feature 2: Redundant (Feature 0 + Feature 1)
# Feature 3: Random data
np.random.seed(42)
f0 = np.random.randint(1, 10, size=(5, 1))
f1 = np.random.randint(1, 10, size=(5, 1))
f2 = f0 + f1  # Purely redundant feature!
f3 = np.random.randint(1, 10, size=(5, 1))

# Combine into a single matrix A (Shape: 5x4)
A = np.hstack([f0, f1, f2, f3]).astype(float)
feature_names = ["Item_Price", "Shipping_Fee", "Total_Cost", "Quantity"]

print("--- Original Dataset Matrix A ---")
print(A)
print(f"Features: {feature_names}\n")

# 2. Compute the RREF to get the Matrix R and find pivot columns
# sympy.Matrix.rref() returns (rref_matrix, pivot_column_indices)
sympy_A = Matrix(A)
R_sympy, pivots = sympy_A.rref()

# Convert sympy Matrix R_sympy back to numpy array
R = np.array(R_sympy).astype(float)

# 3. Construct Matrix C using the exact pivot columns from A
C = A[:, pivots]

print("--- Matrix C (Core Independent Features) ---")
print(C)
print(f"Kept Independent Features: {[feature_names[i] for i in pivots]}\n")

print("--- Matrix R (The Recipes / Dependencies) ---")
print(R)
print(f"Pivot Column Indices: {pivots}\n")

# 4. Verify the decomposition: does C @ R actually reconstruct A?
# R has extra all-zero rows beyond the rank, so trim it down to the
# first `rank` rows before multiplying — otherwise the shapes won't match C.
rank = len(pivots)
R_trimmed = R[:rank, :]
reconstruction = C @ R_trimmed

print("--- Verification: C @ R reconstructs A ---")
print(reconstruction)
print("Matches original A:", np.allclose(A, reconstruction))

# 5. Drop the redundant features
all_indices = set(range(A.shape[1]))
pivot_indices = set(pivots)
redundant_indices = list(all_indices - pivot_indices)

print("--- Dimensionality Reduction Summary ---")
for idx in redundant_indices:
    print(
        f"❌ Deleting feature '{feature_names[idx]}' (Column {idx}) because it is redundant."
    )

# Slice matrix A to only keep the independent features
reduced_A = A[:, pivots]
reduced_feature_names = [feature_names[i] for i in pivots]

print("\n--- Final Reduced Dataset ---")
print(reduced_A)
print(f"Remaining Features: {reduced_feature_names}")